In [1]:
import os
import librosa
import subprocess
import numpy as np

In [2]:
# Constants

SR = 22050  #sampling rate : number of numerical values per second of audio
CHUNK_DURATION = 5
CHUNK_SIZE = SR * CHUNK_DURATION # total number of samples in a chunk
N_FFT = 2048 # number of samples in each FFT window
HOP_LENGTH = 512 # number of samples between successive frames
SPECTROGRAM_DIR_PATH = "../datasets/spectrograms/"

In [3]:
os.makedirs(SPECTROGRAM_DIR_PATH, exist_ok=True)

In [4]:
def load_audio_from_mp4(path): #MUSDB dataset has audio file in mp4 format that's why we need use to ffmpeg else just librosa was fine 

    result = subprocess.run(
        [
            "ffmpeg",
            "-v", "error",
            "-i", path,
            "-f", "f32le",
            "-acodec", "pcm_f32le",
            "-ac", "1",
            "-ar", str(SR),
            "pipe:1"
        ],
        capture_output=True, check=True)

    audio = np.frombuffer(result.stdout, dtype=np.float32)

    return audio

In [5]:
def get_mixture_vocal_instrument_audio_MUSDB():
    audio_files = [os.path.join('../datasets/MUSDB18/stem_files', f) for f in os.listdir('../datasets/MUSDB18/stem_files')]

    vocal_files = {
    os.path.basename(f).replace('.stem_vocals.mp4', ''): f
    for f in audio_files if 'stem_vocals' in f
    }

    instrumental_files = {
        os.path.basename(f).replace('.stem_instrumental.mp4', ''): f
        for f in audio_files if 'stem_instrumental' in f
    }

    common_songs = sorted(vocal_files.keys() & instrumental_files.keys())

    mixture_audios = []
    vocal_audios = []
    instrument_audios = []

    for song in common_songs:

        vocals = load_audio_from_mp4(vocal_files[song])
        instrumental = load_audio_from_mp4(instrumental_files[song])

        length = min(len(vocals), len(instrumental))

        vocals = vocals[:length]
        instrumental = instrumental[:length]

        mixture = vocals + instrumental

        mixture_audios.append(mixture)
        vocal_audios.append(vocals)
        instrument_audios.append(instrumental)
    
    return mixture_audios, vocal_audios, instrument_audios

In [6]:
def get_chunks(signal):

    num_chunks = len(signal)//CHUNK_SIZE

    chunks = [signal[i*CHUNK_SIZE:(i+1)*CHUNK_SIZE] for i in range(num_chunks)]
    return chunks

def compute_spectrogram(signal):

    stft = librosa.stft(signal, n_fft=N_FFT, hop_length=HOP_LENGTH)
    magnitude = np.abs(stft)
    return magnitude

In [ ]:
def MUSDB_pipeline():

    mixture_audios, vocal_audios, instrument_audios = get_mixture_vocal_instrument_audio_MUSDB()

    mixture_chunks = []
    vocal_chunks = []
    instrument_chunks = []

    for i,(mixture_audio, vocal_audio, instrument_audio) in enumerate(zip(mixture_audios, vocal_audios, instrument_audios)):

        mixture_chunks.extend(get_chunks(mixture_audio))
        vocal_chunks.extend(get_chunks(vocal_audio))
        instrument_chunks.extend(get_chunks(instrument_audio))

    for i,(mixture_chunk,vocal_chunk,instrument_chunk) in enumerate(zip(mixture_chunks,vocal_chunks,instrument_chunks)):

        mixture_spectorgram = compute_spectrogram(mixture_chunk)
        vocal_spectorgram = compute_spectrogram(vocal_chunk)
        instrument_spectorgram = compute_spectrogram(instrument_chunk)

        np.save(f"{SPECTROGRAM_DIR_PATH}_MUSDB_{i}_mixture.npy", mixture_spectorgram)
        np.save(f"{SPECTROGRAM_DIR_PATH}_MUSDB_{i}_vocals.npy", vocal_spectorgram)
        np.save(f"{SPECTROGRAM_DIR_PATH}_MUSDB_{i}_instrumental.npy", instrument_spectorgram)

In [8]:
MUSDB_pipeline()